In [1]:
 !pip install transformers torch wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=b3315912b43b90b88ac53aa9c4c79dc067e2da0b9cba258566ab2c6dc2d2af69
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [2]:
import torch
import wikipedia
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
MODEL_NAME = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)

In [4]:
#Confidence Heuristic with Corrective RAG (Retrieval Augmented Generation)
def low_confidence(response):
    if len(response.split()) < 5:
        return True
    if is_repetitive(response):
        return True
    if "i don't know" in response.lower():
        return True
    return False

In [5]:
def retrieve_knowledge(query):
    try:
        summary = wikipedia.summary(query, sentences=2)
        return summary
    except:
        return None

In [6]:
#Response Generation
def generate_response(bot_input_ids):
    output = model.generate(
        bot_input_ids,
        max_new_tokens=100,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )
    return output

In [7]:
def clean_input(text):
    return text.strip()

In [8]:
def is_repetitive(text):
    words = text.split()
    return len(words) != len(set(words))

In [9]:
def chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    chat_history_ids = None

    while True:
        user_input = input("User: ")
        user_input = clean_input(user_input)

        if user_input.lower() in ["exit", "quit", "bye"]:
            print("Chatbot: Goodbye!")
            break

        if not user_input:
            print("Chatbot: Please enter something.")
            continue

        # Encode input
        new_input_ids = tokenizer.encode(
            user_input + tokenizer.eos_token,
            return_tensors='pt'
        ).to(device)

        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        bot_input_ids = bot_input_ids[:, -1000:]

        #  Initial Response
        chat_history_ids = generate_response(bot_input_ids)

        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        # Check Confidence
        if low_confidence(response):
            print("Chatbot: Let me check that for you...")

            knowledge = retrieve_knowledge(user_input)

            if knowledge:
                # Augment Prompt
                augmented_input = f"Context: {knowledge}\nUser: {user_input}\nBot:"

                augmented_ids = tokenizer.encode(
                    augmented_input,
                    return_tensors='pt'
                ).to(device)

                #  Regenerate with context
                chat_history_ids = generate_response(augmented_ids)

                response = tokenizer.decode(
                    chat_history_ids[:, augmented_ids.shape[-1]:][0],
                    skip_special_tokens=True
                )

        print(f"Chatbot: {response}")

In [ ]:
chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: Is Elon Musk The CEO of SpaceX?
